# Event Display for High-Energy Clusters

This notebook loads a cluster CSV produced by the AmBe charge clustering workflow, finds events with at least one clustered object with total energy $3 \leq E < 4$ MeV and more than 4 hits, and displays three events.

For each selected event, the highest-energy qualifying cluster is shown as the candidate cluster and any other clustered hits from the same event are overlaid as separate traces with their own legend entries.


In [43]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "utils" / "my_ev_display.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError("Could not locate the repository root from the current working directory")

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from utils.my_ev_display import event_display


In [44]:
input_csv = Path("/pscratch/sd/l/lmlepin/cluster_outputs/source_in_one_trig_32us_window_20260309_070202_clusters.csv")
candidate_energy_min_mev = 3.0
candidate_energy_max_mev = 4.0
candidate_min_hits = 5
n_events_to_display = 5

input_csv


PosixPath('/pscratch/sd/l/lmlepin/cluster_outputs/source_in_one_trig_32us_window_20260309_070202_clusters.csv')

In [45]:
required_columns = {"x", "y", "z", "E", "light_id", "cluster_label", "file_id"}

df = pd.read_csv(input_csv)
df = df.loc[:, ~df.columns.str.startswith("Unnamed")].copy()
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

numeric_columns = ["x", "y", "z", "E", "light_id", "cluster_label", "file_id"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=numeric_columns).copy()
df["light_id"] = df["light_id"].astype(int)
df["cluster_label"] = df["cluster_label"].astype(int)
df["file_id"] = df["file_id"].astype(int)

clustered_df = df[df["cluster_label"] != -1].copy()
print(f"Loaded {len(df):,} hits from {input_csv}")
print(f"Clustered hits available for selection: {len(clustered_df):,}")


Loaded 894,568 hits from /pscratch/sd/l/lmlepin/cluster_outputs/source_in_one_trig_32us_window_20260309_070202_clusters.csv
Clustered hits available for selection: 894,568


In [46]:
def summarize_clusters(cluster_hits_df):
    summary = (
        cluster_hits_df
        .groupby(["file_id", "light_id", "cluster_label"], as_index=False)
        .agg(
            total_E=("E", "sum"),
            n_hits=("E", "size"),
            mean_x=("x", "mean"),
            mean_y=("y", "mean"),
            mean_z=("z", "mean"),
        )
        .sort_values(["total_E", "n_hits"], ascending=[False, False])
        .reset_index(drop=True)
    )
    return summary


def select_candidate_events(cluster_summary, energy_min_mev=3.0, energy_max_mev=4.0, min_hits=5, max_events=3):
    qualifying = cluster_summary[
        (cluster_summary["total_E"] >= energy_min_mev)
        & (cluster_summary["total_E"] < energy_max_mev)
        & (cluster_summary["n_hits"] >= min_hits)
    ].copy()
    if qualifying.empty:
        raise ValueError(
            f"No clusters found with {energy_min_mev} <= total_E < {energy_max_mev} MeV and n_hits >= {min_hits}"
        )

    per_event_candidates = (
        qualifying
        .sort_values(["total_E", "n_hits"], ascending=[False, False])
        .drop_duplicates(subset=["file_id", "light_id"], keep="first")
        .reset_index(drop=True)
    )

    file_counts = (
        per_event_candidates
        .groupby("file_id")
        .size()
        .sort_values(ascending=False)
    )

    if file_counts.empty:
        raise ValueError("No qualifying events remain after per-event candidate selection")

    selected_file_id = int(file_counts.index[0])
    selected = (
        per_event_candidates[per_event_candidates["file_id"] == selected_file_id]
        .sort_values(["total_E", "n_hits"], ascending=[False, False])
        .head(max_events)
        .reset_index(drop=True)
    )

    if len(selected) < max_events:
        raise ValueError(
            f"file_id {selected_file_id} only has {len(selected)} qualifying events with "
            f"{energy_min_mev} <= E < {energy_max_mev} MeV and n_hits >= {min_hits}"
        )

    return qualifying, selected, selected_file_id


def build_hits_array(hits_df, color_column="E"):
    return hits_df[["x", "y", "z", color_column]].to_numpy(dtype=float)


In [47]:
cluster_summary = summarize_clusters(clustered_df)
qualifying_clusters, selected_events, selected_file_id = select_candidate_events(
    cluster_summary,
    energy_min_mev=candidate_energy_min_mev,
    energy_max_mev=candidate_energy_max_mev,
    min_hits=candidate_min_hits,
    max_events=n_events_to_display,
)

print(
    f"Qualifying clusters with {candidate_energy_min_mev} <= total_E < {candidate_energy_max_mev} MeV and n_hits >= {candidate_min_hits}: "
    f"{len(qualifying_clusters)}"
)
print(f"Displaying events from file_id = {selected_file_id}")
display(selected_events)


Qualifying clusters with 3.0 <= total_E < 4.0 MeV and n_hits >= 5: 366
Displaying events from file_id = 47


,file_id,light_id,cluster_label,total_E,n_hits,mean_x,mean_y,mean_z
0,47,1858,1,3.982710,5,-4.968778,40.465792,19.338913
1,47,520,2,3.920033,6,-11.610021,-19.657400,12.743337
2,47,1648,3,3.561792,9,-38.990864,55.631304,11.062113
3,47,358,4,3.341435,9,-15.285409,40.127700,-8.300100
4,47,2119,0,3.307972,6,-27.297827,49.531475,41.841461


In [48]:
other_cluster_palette = [
    "#1f77b4",
    "#2ca02c",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#17becf",
]

for i, candidate in selected_events.iterrows():
    event_mask = (
        (clustered_df["file_id"] == candidate.file_id)
        & (clustered_df["light_id"] == candidate.light_id)
    )
    event_hits = clustered_df[event_mask].copy()

    candidate_hits = event_hits[event_hits["cluster_label"] == candidate.cluster_label].copy()
    other_hits = event_hits[event_hits["cluster_label"] != candidate.cluster_label].copy()

    title = (
        f"Event {i + 1}: file {int(candidate.file_id)}, light_id {int(candidate.light_id)} | "
        f"candidate cluster {int(candidate.cluster_label)} with total E = {candidate.total_E:.2f} MeV"
    )

    fig = event_display(
        build_hits_array(candidate_hits),
        trace_name=(
            f"Candidate cluster {int(candidate.cluster_label)} "
            f"(sum E = {candidate.total_E:.2f} MeV, {int(candidate.n_hits)} hits)"
        ),
        title=title,
        colorbar_title="Hit E [MeV]",
        colorscale="Purples",
        marker_kwargs={"size": 5, "opacity": 0.95, "color": "#7b2cbf", "showscale": False, "line": {"color": "#2b0a3d", "width": 1}},
        show=False,
        return_fig=True,
    )

    other_summary = summarize_clusters(other_hits)
    for j, other_cluster in other_summary.reset_index(drop=True).iterrows():
        other_cluster_hits = other_hits[other_hits["cluster_label"] == other_cluster.cluster_label]
        color = other_cluster_palette[j % len(other_cluster_palette)]

        fig.add_trace(go.Scatter3d(
            x=other_cluster_hits["x"],
            y=other_cluster_hits["y"],
            z=other_cluster_hits["z"],
            mode="markers",
            marker=dict(size=4, color=color, opacity=0.85),
            name=(
                f"Other cluster {int(other_cluster.cluster_label)} "
                f"(sum E = {other_cluster.total_E:.2f} MeV, {int(other_cluster.n_hits)} hits)"
            ),
        ))

    if other_summary.empty:
        print(
            f"Event file {int(candidate.file_id)}, light_id {int(candidate.light_id)} has no additional clustered hits besides the candidate cluster."
        )

    fig.update_layout(
        width=1250,
        height=850,
        margin=dict(l=20, r=220, t=60, b=20),
        legend=dict(
            title="Displayed clusters",
            itemsizing="constant",
            x=1.02,
            y=0.98,
            xanchor="left",
            yanchor="top",
        )
    )

    fig.show()
